# 05 — Final Evaluation, Interpretation and Reporting

**Input:** `train.parquet`, `test.parquet`, `final_pipeline.joblib`, `model_selection.json`

**Output:** `final_metrics.json`, `model_card.md`

**This notebook opens the test set. Once.**

Every look at a test set spends a little of its value. The protection is not willpower, it is ordering: Part 1 writes down exactly what will be computed and what each outcome will mean, before Part 2 reads the file. Deciding what counts as success after seeing the number is how a held-out set quietly becomes a validation set.

Notebook 04 recorded 213 validation queries. That is a lot of selection pressure, so the test figure is expected to come in worse than the cross-validated one. The size of that gap is itself a result worth reporting.

**Parts**

1. Pre-registration
2. Single test evaluation, with intervals
3. Confusion and error analysis
4. Slice evaluation, including fairness
5. Interpretation: coefficients, permutation importance, SHAP
6. Per-prediction uncertainty via conformal prediction
7. Sensitivity: what the conclusion depends on
8. Model card

---

## Setup

In [ ]:
# Restart the kernel after this cell before running the rest.
%pip install -q --force-reinstall --no-deps git+https://github.com/rbennum/telco-churn.git
%pip install -q pyarrow joblib shap

In [ ]:
import json
import warnings
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from telco_churn.config import RANDOM_SEED, TARGET, OUT_DIR, ensure_out_dir, find_artifact
from telco_churn.business import (
    COST_ASSUMPTIONS,
    breakeven_effectiveness,
    oracle_cost,
    oracle_policy,
    cost_per_customer,
    offer_cost,
    optimal_threshold,
    policy_from_proba,
    value_at_risk,
)

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110

ensure_out_dir()

selection = json.loads(find_artifact("model_selection.json").read_text())
manifest = json.loads(find_artifact("ingest_manifest.json").read_text())
pipeline = joblib.load(find_artifact("final_pipeline.joblib"))

print(f"model      : {selection['chosen_model']}")
print(f"calibration: {selection['chosen_calibration']}")
print(f"threshold  : {selection['chosen_threshold_strategy']}")
print(f"cv cost    : USD {selection['final_config_cv_cost']:.4f}")
print(f"cv verdict : {selection['verdict']}")
print(f"queries    : {selection['validation_queries']}")

---

## 1. Pre-Registration

Written before the test set is read. Each line commits to a computation and to what its outcome will mean, so that no result can be reinterpreted after the fact to look better than it is.

**Primary metric.** Expected cost per customer on the test set, using the derived per-customer threshold $p^*_i = C_i / (\varepsilon V_i)$. No threshold will be re-fitted on test data.

**Decision rule, unchanged from notebook 01.** Headroom captured against the test-set oracle and the test-set best naive policy. Continue at ≥40%, abandon below 15%. The cross-validated estimate was 23.6% with a 95% interval of [10.5%, 36.9%], and the verdict was DOES NOT CLEAR.

**What each outcome will mean.**

If the test figure lands inside the cross-validated interval, the estimate held and the verdict stands. If it lands materially below, the 213 validation queries bought optimism and the honest reading is that the search overfit the folds. If it lands above, that is favourable noise and will be reported as noise, not as evidence the model is better than measured.

**Secondary quantities**, all reported regardless of what they show: ROC-AUC, average precision, log loss, Brier, MCC, a stratified bootstrap interval on each, per-slice performance including protected attributes, permutation importance on test data, conformal coverage at 90%, and a sensitivity analysis over $\varepsilon$ and gross margin.

**Committed in advance: no iteration after this point.** If the result is disappointing, it is reported as it is. Any further modelling would make this an in-sample estimate and would have to be declared as such.

In [ ]:
PASS_AT, ABORT_BELOW = 0.40, 0.15
ALPHA_CONFORMAL = 0.10          # 90% coverage
CV_ESTIMATE = selection["headroom_captured_point"]
CV_INTERVAL = selection["headroom_captured_ci"]

prereg = {
    "primary_metric": "expected cost per customer, derived per-customer threshold",
    "threshold_strategy": selection["chosen_threshold_strategy"],
    "threshold_refit_on_test": False,
    "decision_rule": {"continue_at_or_above": PASS_AT, "abandon_below": ABORT_BELOW},
    "cv_estimate": CV_ESTIMATE,
    "cv_interval": CV_INTERVAL,
    "cv_verdict": selection["verdict"],
    "validation_queries_spent": selection["validation_queries"],
    "iteration_after_this_point": "none — any further modelling makes this in-sample",
    "registered_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
(OUT_DIR / "preregistration.json").write_text(json.dumps(prereg, indent=2))
print(json.dumps(prereg, indent=2))

---

## 2. The Single Test Evaluation

In [ ]:
test = pd.read_parquet(find_artifact("test.parquet"))
train = pd.read_parquet(find_artifact("train.parquet"))

assert len(test) == manifest["test_rows"], "Stale test.parquet"
assert list(test.columns) == list(train.columns), "Column mismatch across split"

X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]
mc_test, ct_test = X_test["MonthlyCharges"], X_test["Contract"]

print(f"test rows      : {len(test)}")
print(f"test churn rate: {y_test.mean():.4f}")
print(f"train churn    : {train[TARGET].mean():.4f}")

proba = pipeline.predict_proba(X_test)[:, 1]
thresholds = optimal_threshold(mc_test, ct_test)
action = policy_from_proba(proba, mc_test, ct_test)

print(f"\ncontacted      : {action.mean():.1%} of the test base")
print(f"mean p*        : {thresholds.mean():.4f}")

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    matthews_corrcoef,
    roc_auc_score,
)

# Benchmarks recomputed ON THE TEST SET. Reusing the training-set oracle would
# compare against a ceiling from a different sample.
oracle_test = oracle_cost(y_test, mc_test, ct_test)
naive_nobody = cost_per_customer(y_test, np.zeros(len(y_test)), mc_test, ct_test)
naive_m2m = cost_per_customer(y_test, (ct_test == "Month-to-month").astype(float),
                              mc_test, ct_test)
best_naive_test = min(naive_nobody, naive_m2m)
headroom_test = best_naive_test - oracle_test

model_cost = cost_per_customer(y_test, action, mc_test, ct_test)
captured = (best_naive_test - model_cost) / headroom_test

test_metrics = {
    "roc_auc": roc_auc_score(y_test, proba),
    "average_precision": average_precision_score(y_test, proba),
    "log_loss": log_loss(y_test, proba),
    "brier": brier_score_loss(y_test, proba),
    "mcc": matthews_corrcoef(y_test, action),
    "cost_per_customer": model_cost,
    "contacted_share": float(action.mean()),
}
display(pd.Series(test_metrics).round(5).to_frame("test"))

print(f"\noracle (test)      USD {oracle_test:7.3f}")
print(f"best naive (test)  USD {best_naive_test:7.3f}   "
      f"({'contact nobody' if naive_nobody < naive_m2m else 'contact month-to-month'})")
print(f"model (test)       USD {model_cost:7.3f}")
print(f"headroom (test)    USD {headroom_test:7.3f}")
print(f"\nHEADROOM CAPTURED  {captured:.1%}")

In [ ]:
from sklearn.utils import resample

rng = np.random.RandomState(RANDOM_SEED)
idx = np.arange(len(y_test))
boot = {"roc_auc": [], "average_precision": [], "cost": [], "captured": []}

for _ in range(2000):
    b = resample(idx, stratify=y_test, random_state=rng.randint(1_000_000))
    yb, pb = y_test.iloc[b], proba[b]
    mcb, ctb = mc_test.iloc[b], ct_test.iloc[b]
    ab = policy_from_proba(pb, mcb, ctb)
    c_model = cost_per_customer(yb, ab, mcb, ctb)
    c_oracle = oracle_cost(yb, mcb, ctb)
    c_naive = min(cost_per_customer(yb, np.zeros(len(yb)), mcb, ctb),
                  cost_per_customer(yb, (ctb == "Month-to-month").astype(float), mcb, ctb))
    boot["roc_auc"].append(roc_auc_score(yb, pb))
    boot["average_precision"].append(average_precision_score(yb, pb))
    boot["cost"].append(c_model)
    boot["captured"].append((c_naive - c_model) / (c_naive - c_oracle))

ci = pd.DataFrame({k: {"mean": np.mean(v),
                       "lo_2.5": np.percentile(v, 2.5),
                       "hi_97.5": np.percentile(v, 97.5)}
                   for k, v in boot.items()}).T
display(ci.round(4))

cap_lo, cap_hi = np.percentile(boot["captured"], [2.5, 97.5])

In [ ]:
if cap_lo >= PASS_AT:
    VERDICT, note = "PASS", "the whole interval clears the continue threshold."
elif cap_hi < ABORT_BELOW:
    VERDICT, note = "ABANDON", "the whole interval sits below the abandon threshold."
elif cap_hi < PASS_AT:
    VERDICT, note = "DOES NOT CLEAR", (
        "the interval never reaches the continue threshold set before modelling began.")
else:
    VERDICT, note = "INCONCLUSIVE", (
        "the interval spans the decision boundary and cannot settle the question.")

drift = captured - CV_ESTIMATE
inside = CV_INTERVAL[0] <= captured <= CV_INTERVAL[1]

print(f"cross-validated : {CV_ESTIMATE:.1%}  CI [{CV_INTERVAL[0]:.1%}, {CV_INTERVAL[1]:.1%}]")
print(f"test            : {captured:.1%}  CI [{cap_lo:.1%}, {cap_hi:.1%}]")
print(f"difference      : {drift:+.1%}")
print(f"inside the cross-validated interval: {inside}")

print(f"\nVERDICT: {VERDICT}")
print(note)

if not inside and drift < 0:
    print("\nThe test figure fell below the cross-validated interval. With 213 "
          "validation queries spent, the most likely explanation is selection "
          "pressure on the folds rather than anything about the test sample.")
elif inside:
    print("\nThe test figure landed inside the cross-validated interval, so the "
          "validation procedure estimated honestly despite the query count.")

---

## 3. Confusion and Error Analysis

Aggregate metrics average away exactly the information needed to improve a model. Reading the confidently wrong cases produces better ideas than another tuning round, and the largest error category is usually a data problem rather than a model problem.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(y_test, action)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
ConfusionMatrixDisplay(cm, display_labels=["Stay", "Churn"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Counts")
ConfusionMatrixDisplay(confusion_matrix(y_test, action, normalize="true"),
                       display_labels=["Stay", "Churn"]).plot(
    ax=axes[1], colorbar=False, values_format=".3f")
axes[1].set_title("Row-normalised (per-class recall)")
plt.tight_layout(); plt.show()

v = value_at_risk(mc_test, ct_test)
c = offer_cost(mc_test)
eps = COST_ASSUMPTIONS["offer_effectiveness"]

econ = pd.DataFrame([
    {"outcome": "TN  stayed, not contacted", "n": tn, "unit_cost": 0.0},
    {"outcome": "FP  stayed, contacted", "n": fp, "unit_cost": c[(y_test == 0) & (action == 1)].mean() if fp else 0},
    {"outcome": "FN  churned, not contacted", "n": fn, "unit_cost": v[(y_test == 1) & (action == 0)].mean() if fn else 0},
    {"outcome": "TP  churned, contacted", "n": tp,
     "unit_cost": ((1 - eps) * v + c)[(y_test == 1) & (action == 1)].mean() if tp else 0},
])
econ["total_cost"] = econ["n"] * econ["unit_cost"]
econ["share_%"] = 100 * econ["total_cost"] / econ["total_cost"].sum()
display(econ.round(2))

print("Read the share column, not the count column. Where the money actually goes")
print("is rarely where the error count is largest.")

In [ ]:
errors = X_test.copy()
errors["y_true"] = y_test.values
errors["proba"] = proba
errors["p_star"] = thresholds
errors["action"] = action
errors["value_at_risk"] = v
errors["cost_incurred"] = np.where(
    (y_test == 1) & (action == 0), v,
    np.where((y_test == 1) & (action == 1), (1 - eps) * v + c,
             np.where((y_test == 0) & (action == 1), c, 0.0)))
errors["error_type"] = np.select(
    [(y_test == 1) & (action == 0), (y_test == 0) & (action == 1)],
    ["FN", "FP"], default="correct")

wrong = errors.query("error_type != 'correct'").copy()
wrong["confidence"] = (wrong["proba"] - wrong["p_star"]).abs()

print("Ten most expensive errors:")
display(wrong.nlargest(10, "cost_incurred")[
    ["error_type", "proba", "p_star", "tenure", "Contract", "InternetService",
     "MonthlyCharges", "value_at_risk", "cost_incurred"]].round(3))

print("\nTen most confidently wrong (furthest past their own threshold):")
display(wrong.nlargest(10, "confidence")[
    ["error_type", "proba", "p_star", "confidence", "tenure", "Contract",
     "PaymentMethod", "MonthlyCharges"]].round(3))

In [ ]:
# Where do errors concentrate? Categories here are structural rather than
# hand-labelled, which is the cheap version of the manual triage the full
# procedure calls for.
profile = wrong.groupby(["error_type", "Contract"], observed=False).agg(
    n=("proba", "size"),
    mean_proba=("proba", "mean"),
    total_cost=("cost_incurred", "sum"),
).reset_index()
profile["cost_share_%"] = 100 * profile["total_cost"] / errors["cost_incurred"].sum()
display(profile.round(2).sort_values("total_cost", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for et, grp in wrong.groupby("error_type"):
    axes[0].hist(grp["proba"], bins=30, alpha=0.6, label=et)
axes[0].axvline(thresholds.mean(), color="crimson", ls="--", lw=1, label="mean p*")
axes[0].set_xlabel("predicted probability"); axes[0].legend()
axes[0].set_title("Where errors sit in probability space")

axes[1].scatter(errors["proba"], errors["value_at_risk"],
                c=(errors["error_type"] != "correct"), cmap="coolwarm",
                alpha=0.35, s=12)
axes[1].axvline(thresholds.mean(), color="grey", ls="--", lw=1)
axes[1].set_xlabel("predicted probability"); axes[1].set_ylabel("value at risk (USD)")
axes[1].set_title("Errors against what they cost")
plt.tight_layout(); plt.show()

---

## 4. Slice Evaluation

A model with strong overall performance can be unusable for one segment, and aggregate numbers hide that completely.

Two points matter for reading this table honestly. Base rate must be reported next to every metric, because a slice with a different base rate is not comparable on precision-style measures, and mistaking that for a fairness problem is a common misreading. And calibration must be checked per slice as well as discrimination, since a model can rank correctly inside every subgroup while systematically over-predicting risk for one of them — invisible in AUC, highly visible to whoever is affected.

**Fairness criteria conflict mathematically.** Equal calibration across groups and equal error rates cannot generally both hold when base rates differ. The choice below is calibration parity, on the grounds that the probability feeds a monetary decision. That is a choice, stated rather than assumed.

In [ ]:
def slice_report(frame, y_true, p, act, key):
    rows = []
    for level, g in frame.groupby(key, observed=False):
        m = g.index
        yt, pp, aa = y_true.loc[m], p[frame.index.get_indexer(m)], act[frame.index.get_indexer(m)]
        if yt.nunique() < 2:
            continue
        mcg, ctg = frame.loc[m, "MonthlyCharges"], frame.loc[m, "Contract"]
        rows.append({
            "slice": f"{key}={level}",
            "n": len(g),
            "base_rate": yt.mean(),
            "mean_pred": pp.mean(),
            "calib_gap": pp.mean() - yt.mean(),
            "roc_auc": roc_auc_score(yt, pp),
            "avg_prec": average_precision_score(yt, pp),
            "brier": brier_score_loss(yt, pp),
            "contacted": aa.mean(),
            "cost": cost_per_customer(yt, aa, mcg, ctg),
        })
    return pd.DataFrame(rows)


frame = X_test.reset_index(drop=True)
y_reset = y_test.reset_index(drop=True)

SLICE_KEYS = ["gender", "SeniorCitizen", "Partner", "Dependents",
              "Contract", "InternetService", "PaymentMethod"]

slices = pd.concat([slice_report(frame, y_reset, proba, action, k) for k in SLICE_KEYS],
                   ignore_index=True)
display(slices.round(4).sort_values("brier", ascending=False))

In [ ]:
# Brier rewards low base rates, so ranking by it hides the slices where the model
# is actually weakest at DISCRIMINATING. Rank by ROC-AUC against the overall figure.
overall_auc = roc_auc_score(y_test, proba)
weak = slices.assign(auc_gap=slices["roc_auc"] - overall_auc).nsmallest(6, "roc_auc")

print(f"overall ROC-AUC: {overall_auc:.4f}\n")
display(weak[["slice", "n", "base_rate", "roc_auc", "auc_gap", "avg_prec", "calib_gap"]]
        .round(4).reset_index(drop=True))

worst_auc = weak.iloc[0]
print(f"\nWeakest slice: {worst_auc['slice']} at ROC-AUC {worst_auc['roc_auc']:.4f}, "
      f"{overall_auc - worst_auc['roc_auc']:.4f} below overall (n={int(worst_auc['n'])}).")
print("\nA slice this far below the headline figure is a finding to report, not a")
print("rounding detail. Note also that average precision falls with the base rate")
print("by construction, so a low figure in a low-churn slice is expected and is")
print("not evidence of unfairness — compare ROC-AUC and calibration gap instead.")

In [ ]:
PROTECTED = ["gender", "SeniorCitizen", "Partner", "Dependents"]
prot = slices[slices["slice"].str.split("=").str[0].isin(PROTECTED)].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
axes[0].barh(prot["slice"], prot["calib_gap"])
axes[0].axvline(0, color="black", lw=1)
axes[0].set_xlabel("mean predicted minus observed churn rate")
axes[0].set_title("Calibration gap by protected attribute")
axes[0].invert_yaxis()

axes[1].scatter(prot["base_rate"], prot["contacted"], s=prot["n"] / 4 + 20)
for _, r in prot.iterrows():
    axes[1].annotate(r["slice"], (r["base_rate"], r["contacted"]), fontsize=7)
lims = [0, max(prot["base_rate"].max(), prot["contacted"].max()) * 1.15]
axes[1].plot(lims, lims, "k--", lw=1)
axes[1].set_xlabel("observed churn rate"); axes[1].set_ylabel("share contacted")
axes[1].set_title("Contact rate against actual risk")
plt.tight_layout(); plt.show()

worst = prot.loc[prot["calib_gap"].abs().idxmax()]
print(f"largest calibration gap: {worst['slice']} at {worst['calib_gap']:+.4f} "
      f"(n={int(worst['n'])}, base rate {worst['base_rate']:.3f})")
print("\nA gap below roughly 0.02 is small relative to the sampling noise in a slice")
print("this size. Report the number either way rather than declaring parity.")

---

## 5. Interpretation

Three views, because each answers a different question. Coefficients say what the model does. Permutation importance on held-out data says what actually matters for performance. SHAP says how a specific prediction was reached.

**Correlated features distort all three.** `TotalCharges` had a VIF of 9.33 against `tenure` and `MonthlyCharges`, so credit gets divided between them in ways that are easy to over-interpret. Read that group together, not separately.

In [ ]:
# Unwrap whatever configuration notebook 04 selected, so this works whether or
# not a calibration wrapper was applied.
inner = pipeline
if hasattr(pipeline, "calibrated_classifiers_"):
    inner = pipeline.calibrated_classifiers_[0].estimator
prep = inner.named_steps["prep"]
model = inner.named_steps["model"]

feature_names = list(prep.get_feature_names_out())

if hasattr(model, "coef_"):
    coefs = pd.DataFrame({
        "feature": feature_names,
        "coefficient": model.coef_[0],
        "odds_ratio": np.exp(model.coef_[0]),
    }).sort_values("coefficient", key=abs, ascending=False)
    display(coefs.head(15).round(4))

    fig, ax = plt.subplots(figsize=(9, 6))
    top = coefs.head(18).iloc[::-1]
    ax.barh(top["feature"], top["coefficient"],
            color=np.where(top["coefficient"] > 0, "firebrick", "steelblue"))
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("coefficient (log-odds, standardised inputs)")
    ax.set_title("Direction and magnitude — red raises churn odds")
    plt.tight_layout(); plt.show()
    print("Numeric features are standardised, so their coefficients are per standard")
    print("deviation. Categorical coefficients are against the dropped reference level.")
else:
    coefs = None
    print("Model has no linear coefficients; rely on permutation importance and SHAP.")

In [ ]:
# Computed directly on the DataFrame. sklearn's permutation_importance converts
# the frame to a bare array internally, which strips column names and makes every
# transformer in the pipeline emit a feature-names warning. Doing it by hand keeps
# the frame intact, and also allows permuting a GROUP of columns together.


def permutation_importance_df(estimator, frame, target, n_repeats=20,
                              groups=None, seed=RANDOM_SEED):
    """Increase in log loss when a feature (or a group of features) is shuffled.

    Permuting a correlated group with the SAME permutation preserves the
    relationships inside the group while severing its link to the target, which
    measures the group's joint contribution. Permuting members separately splits
    credit between them in ways that are easy to misread.
    """
    rng = np.random.RandomState(seed)
    base = -log_loss(target, estimator.predict_proba(frame)[:, 1])
    items = groups if groups else {c: [c] for c in frame.columns}

    rows = []
    for name, cols in items.items():
        drops = []
        for _ in range(n_repeats):
            perm = rng.permutation(len(frame))
            shuffled = frame.copy()
            for c in cols:
                shuffled[c] = frame[c].to_numpy()[perm]
            drops.append(base - (-log_loss(target, estimator.predict_proba(shuffled)[:, 1])))
        rows.append({"feature": name, "importance": np.mean(drops), "sd": np.std(drops)})

    return pd.DataFrame(rows).sort_values("importance", ascending=False).reset_index(drop=True)


imp = permutation_importance_df(pipeline, X_test, y_test, n_repeats=20)
display(imp.round(5).head(15))

fig, ax = plt.subplots(figsize=(9, 5.6))
top = imp.head(15).iloc[::-1]
ax.barh(top["feature"], top["importance"], xerr=top["sd"])
ax.set_xlabel("increase in log loss when shuffled")
ax.set_title("Permutation importance, measured on the test set")
plt.tight_layout(); plt.show()

print("Measured on held-out data, so this answers how much performance genuinely")
print("depends on each feature — a stricter question than any in-sample measure.")

### Grouped importance for the correlated block

`TotalCharges` had a VIF of 9.33 against `tenure` and `MonthlyCharges`. When
features are correlated, permuting one at a time understates each, because the
model recovers most of the lost signal from its neighbours. Permuting the whole
block together measures what the group jointly contributes.

In [ ]:
GROUPS = {
    "charges_block (tenure+MonthlyCharges+TotalCharges)":
        ["tenure", "MonthlyCharges", "TotalCharges"],
    "tenure_alone": ["tenure"],
    "TotalCharges_alone": ["TotalCharges"],
    "MonthlyCharges_alone": ["MonthlyCharges"],
    "contract_and_internet": ["Contract", "InternetService"],
    "addon_services": ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                       "TechSupport", "StreamingTV", "StreamingMovies"],
}

grouped = permutation_importance_df(pipeline, X_test, y_test, n_repeats=10, groups=GROUPS)
display(grouped.round(5))

block = grouped.set_index("feature").loc[
    "charges_block (tenure+MonthlyCharges+TotalCharges)", "importance"]
parts = grouped.set_index("feature").loc[
    ["tenure_alone", "TotalCharges_alone", "MonthlyCharges_alone"], "importance"].sum()
ratio = block / parts
print(f"\nblock permuted together : {block:.5f}")
print(f"sum of individual drops : {parts:.5f}")
print(f"ratio                   : {ratio:.2f}x")

if ratio > 1.1:
    print("\nRatio above 1: MASKING. Individually each feature looks dispensable")
    print("because its neighbours carry the same signal, so the solo figures")
    print("understate what the group jointly contributes.")
elif ratio < 0.9:
    print("\nRatio below 1: DOUBLE-COUNTING. Shuffling one member alone leaves the")
    print("others inconsistent with it, and that contradiction is itself a signal the")
    print("model reacts to. So each solo figure charges part of the same shared")
    print("information to a different feature, and the individual numbers sum to more")
    print("than the group is actually worth.")
    print(f"\nThe group's real joint contribution is {block:.5f}. Report that, not the")
    print("individual figures, when describing how much the model depends on billing")
    print("history — the solo numbers are inflated by the correlation, not by signal.")
else:
    print("\nRatio near 1: the features contribute roughly independently despite")
    print("their correlation, so the individual figures can be read at face value.")

In [ ]:
try:
    import shap

    X_bg = prep.transform(train.drop(columns=[TARGET]).sample(
        min(500, len(train)), random_state=RANDOM_SEED))
    X_shap = prep.transform(X_test)

    if hasattr(model, "coef_"):
        explainer = shap.LinearExplainer(model, X_bg)
    else:
        explainer = shap.Explainer(model, X_bg)
    shap_values = explainer(X_shap)

    shap.summary_plot(shap_values, X_shap, feature_names=feature_names,
                      max_display=15, show=False)
    plt.tight_layout(); plt.show()

    shap_imp = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": np.abs(shap_values.values).mean(0),
    }).sort_values("mean_abs_shap", ascending=False)
    display(shap_imp.head(12).round(5))
    SHAP_OK = True
except Exception as exc:
    SHAP_OK = False
    shap_imp = None
    print(f"SHAP unavailable ({type(exc).__name__}: {exc}).")
    print("Coefficients and permutation importance above already cover the reporting")
    print("requirement; SHAP adds per-prediction attribution, not new global insight.")

In [ ]:
if SHAP_OK:
    # Explain one expensive mistake. A single case is what a retention agent
    # actually sees, and it is the level at which the model has to be defensible.
    worst_idx = int(wrong.nlargest(1, "cost_incurred").index[0])
    pos = X_test.index.get_loc(worst_idx)
    shap.plots.waterfall(shap_values[pos], max_display=12, show=False)
    plt.tight_layout(); plt.show()
    print(X_test.loc[worst_idx, ["tenure", "Contract", "InternetService",
                                 "MonthlyCharges", "PaymentMethod"]].to_string())
    print(f"\npredicted {proba[pos]:.4f} | threshold {thresholds[pos]:.4f} | "
          f"actual {'churned' if y_test.loc[worst_idx] else 'stayed'}")

---

## 6. Per-Prediction Uncertainty

The bootstrap in Part 2 quantifies uncertainty in the *metric*. It says nothing about uncertainty in an *individual* prediction, which is what a retention agent working a single case actually needs.

Split conformal prediction supplies that, with a distribution-free finite-sample coverage guarantee. The implementation below is the LAC score written out directly rather than pulled from a library, because it is roughly ten lines and avoids a dependency whose API has churned.

**Calibration data.** The pipeline is already fitted on the full training set, so training rows would give in-sample scores and break the guarantee. Out-of-fold predictions on training data are used instead, which are genuinely out-of-sample while leaving the test set untouched.

**The guarantee is marginal, not conditional.** Coverage holds on average across all cases, not within every subgroup, so per-slice coverage is checked below.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X_train_full = train.drop(columns=[TARGET])
y_train_full = train[TARGET]

cal_proba = cross_val_predict(
    pipeline, X_train_full, y_train_full,
    cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_SEED),
    method="predict_proba", n_jobs=-1,
)

# LAC nonconformity: 1 - predicted probability of the TRUE class.
cal_scores = 1.0 - cal_proba[np.arange(len(y_train_full)), y_train_full.to_numpy()]

n = len(cal_scores)
level = np.ceil((n + 1) * (1 - ALPHA_CONFORMAL)) / n
qhat = float(np.quantile(cal_scores, min(level, 1.0), method="higher"))

test_proba_2col = np.column_stack([1 - proba, proba])
pred_sets = test_proba_2col >= (1 - qhat)
set_sizes = pred_sets.sum(axis=1)
covered = pred_sets[np.arange(len(y_test)), y_test.to_numpy()]

print(f"calibration rows : {n}")
print(f"qhat at alpha={ALPHA_CONFORMAL} : {qhat:.4f}")
print(f"target coverage  : {1 - ALPHA_CONFORMAL:.0%}")
print(f"observed coverage: {covered.mean():.1%}")
print(f"\nset size distribution:")
for s, cnt in zip(*np.unique(set_sizes, return_counts=True)):
    label = {0: "empty — model confident but score below threshold",
             1: "decisive", 2: "ambiguous, route to a human"}[int(s)]
    print(f"  size {s}: {cnt:5d} ({cnt/len(set_sizes):6.1%})  {label}")

In [ ]:
conf = pd.DataFrame({
    "set_size": set_sizes, "covered": covered, "proba": proba,
    "Contract": ct_test.values, "y_true": y_test.values,
})

per_slice_cov = conf.groupby("Contract", observed=False).agg(
    n=("covered", "size"), coverage=("covered", "mean"),
    ambiguous=("set_size", lambda s: (s == 2).mean()),
).round(4)
display(per_slice_cov)

fig, ax = plt.subplots(figsize=(9, 4))
for s in sorted(conf["set_size"].unique()):
    ax.hist(conf.loc[conf["set_size"] == s, "proba"], bins=30, alpha=0.6,
            label=f"set size {s}")
ax.set_xlabel("predicted probability"); ax.legend()
ax.set_title("Ambiguous cases cluster where the model is genuinely undecided")
plt.tight_layout(); plt.show()

print("Marginal coverage holds overall by construction. Per-slice coverage is not")
print("guaranteed, and a slice materially below target is a finding to report.\n")

target = 1 - ALPHA_CONFORMAL
short = per_slice_cov[per_slice_cov["coverage"] < target - 0.02]
if len(short):
    for name, r in short.iterrows():
        print(f"UNDER TARGET: {name} covered {r['coverage']:.1%} against {target:.0%} "
              f"(n={int(r['n'])}, {r['ambiguous']:.1%} ambiguous)")
    print("\nThis is the marginal-versus-conditional distinction made concrete. The")
    print("guarantee was never a promise about subgroups, and the shortfall lands on")
    print("the segment carrying most of the churn — the one an operator cares about.")
    print("Fix by calibrating conformal scores WITHIN each contract type, which buys")
    print("per-group coverage at the cost of wider sets.")
else:
    print(f"All slices within 2 points of the {target:.0%} target.")

---

## 7. Sensitivity: What the Conclusion Rests On

The cost model contains one parameter that does not appear anywhere in the data and cannot be estimated from it: $\varepsilon$, the probability an offer actually retains a would-be churner. Notebook 01 fixed it at 0.30 by assumption.

Everything downstream inherits that assumption. The threshold, the headroom, and the verdict all move with it. This section maps how far.

**Why this is the most important analysis in the project.** If the verdict flips across a plausible range of $\varepsilon$, then the business case is decided by a number nobody has measured, and the correct recommendation is to run a randomised retention experiment before spending anything further on modelling. That conclusion would be worth more than any improvement in AUC.

In [ ]:
def evaluate_at(eps=None, margin=None, proba_vec=proba):
    a = dict(COST_ASSUMPTIONS)
    if eps is not None:
        a["offer_effectiveness"] = eps
    if margin is not None:
        a["gross_margin"] = margin

    act = policy_from_proba(proba_vec, mc_test, ct_test, a)
    m = cost_per_customer(y_test, act, mc_test, ct_test, a)
    # Cost-aware oracle: contact a known churner only when eps*V > C. Contacting
    # every churner regardless is not a ceiling — at low eps it costs more than
    # doing nothing, which yields negative headroom and a meaningless ratio.
    o = oracle_cost(y_test, mc_test, ct_test, a)
    nb = min(cost_per_customer(y_test, np.zeros(len(y_test)), mc_test, ct_test, a),
             cost_per_customer(y_test, (ct_test == "Month-to-month").astype(float),
                               mc_test, ct_test, a))
    hd = nb - o
    # hd == 0 means perfect foresight is worth nothing: no customer clears
    # break-even, so targeting cannot help at any skill level.
    return {"model": m, "oracle": o, "best_naive": nb, "headroom": hd,
            "captured": (nb - m) / hd if hd > 1e-9 else 0.0,
            "targeting_viable": hd > 1e-9,
            "contacted": float(act.mean())}


eps_grid = np.round(np.arange(0.05, 0.66, 0.05), 3)
sens = pd.DataFrame([{"epsilon": e, **evaluate_at(eps=e)} for e in eps_grid])
sens["verdict"] = np.where(~sens["targeting_viable"], "NO PRIZE",
                    np.where(sens["captured"] >= PASS_AT, "PASS",
                      np.where(sens["captured"] < ABORT_BELOW, "ABANDON",
                               "DOES NOT CLEAR")))
display(sens.round(4))

print("NO PRIZE means headroom is zero: offers are unprofitable even for customers")
print("you KNOW will churn, so no model of any quality changes the outcome.")

flip = sens.loc[sens["captured"] >= PASS_AT, "epsilon"]
print(f"\nassumed epsilon      : {COST_ASSUMPTIONS['offer_effectiveness']}")
print(f"epsilon needed to PASS: "
      f"{flip.min() if len(flip) else 'not reached within the grid'}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))

axes[0].plot(sens["epsilon"], sens["captured"], "o-")
axes[0].axhline(PASS_AT, color="green", ls="--", lw=1, label="continue")
axes[0].axhline(ABORT_BELOW, color="crimson", ls="--", lw=1, label="abandon")
axes[0].axvline(COST_ASSUMPTIONS["offer_effectiveness"], color="grey", ls=":", lw=1.2,
                label="assumed")
axes[0].set_xlabel("offer effectiveness (epsilon)"); axes[0].set_ylabel("headroom captured")
axes[0].set_title("The verdict as a function of an unmeasured parameter")
axes[0].legend(fontsize=8)

axes[1].plot(sens["epsilon"], sens["headroom"], "o-", label="headroom (USD)")
axes[1].plot(sens["epsilon"], sens["best_naive"] - sens["model"], "o-",
             label="model gain (USD)")
axes[1].set_xlabel("epsilon"); axes[1].set_ylabel("USD per customer")
axes[1].set_title("The prize itself scales with epsilon")
axes[1].legend(fontsize=8)

margins = np.round(np.arange(0.20, 0.61, 0.05), 2)
grid = np.array([[evaluate_at(eps=e, margin=m)["captured"] for m in margins]
                 for e in eps_grid])
im = axes[2].imshow(grid, aspect="auto", origin="lower", cmap="RdYlGn",
                    vmin=0, vmax=0.6,
                    extent=[margins[0], margins[-1], eps_grid[0], eps_grid[-1]])
axes[2].set_xlabel("gross margin"); axes[2].set_ylabel("epsilon")
axes[2].set_title("Headroom captured across both assumptions")
plt.colorbar(im, ax=axes[2])
plt.tight_layout(); plt.show()

print("If the green region begins near the assumed values, the conclusion is")
print("fragile and the next spend should be a randomised experiment, not a model.")

In [ ]:
# Break-even: below this epsilon, contacting a certain churner loses money.
be = breakeven_effectiveness(mc_test, ct_test)
display(pd.DataFrame({"Contract": ct_test.values, "breakeven_eps": be})
        .groupby("Contract", observed=False)["breakeven_eps"]
        .agg(["count", "mean", "min", "max"]).round(4))

print(f"\nassumed epsilon        : {COST_ASSUMPTIONS['offer_effectiveness']:.2f}")
print(f"highest break-even     : {be.max():.4f}")
print(f"customers viable at the assumed value: {(be < COST_ASSUMPTIONS['offer_effectiveness']).mean():.1%}")
print("\nBreak-even answers whether the campaign works at all. The sensitivity above")
print("answers whether the MODEL earns its keep. They are different questions and")
print("the project can fail either one independently.")

---

## 8. Model Card

In [ ]:
final_metrics = {
    "notebook": "05_final_evaluation",
    "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "random_seed": RANDOM_SEED,
    "model": {
        "estimator": selection["chosen_model"],
        "calibration": selection["chosen_calibration"],
        "threshold_strategy": selection["chosen_threshold_strategy"],
        "params": selection.get("best_params", {}),
    },
    "data": {
        "train_rows": manifest["train_rows"],
        "test_rows": manifest["test_rows"],
        "n_features": manifest["n_features"],
        "test_positive_rate": round(float(y_test.mean()), 5),
    },
    "test_metrics": {k: round(float(v), 5) for k, v in test_metrics.items()},
    "bootstrap_ci": ci.round(5).to_dict("index"),
    "benchmarks": {
        "oracle": round(float(oracle_test), 4),
        "best_naive": round(float(best_naive_test), 4),
        "headroom": round(float(headroom_test), 4),
    },
    "headroom_captured": {
        "point": round(float(captured), 4),
        "ci": [round(float(cap_lo), 4), round(float(cap_hi), 4)],
        "cv_estimate": CV_ESTIMATE,
        "cv_interval": CV_INTERVAL,
        "test_minus_cv": round(float(drift), 4),
        "inside_cv_interval": bool(inside),
    },
    "verdict": VERDICT,
    "verdict_note": note,
    "validation_queries_spent": selection["validation_queries"],
    "conformal": {
        "alpha": ALPHA_CONFORMAL,
        "qhat": round(qhat, 4),
        "observed_coverage": round(float(covered.mean()), 4),
        "ambiguous_share": round(float((set_sizes == 2).mean()), 4),
        "empty_share": round(float((set_sizes == 0).mean()), 4),
    },
    "sensitivity": sens.round(4).to_dict("records"),
    "cost_assumptions": COST_ASSUMPTIONS,
    "slices": slices.round(4).to_dict("records"),
}

(OUT_DIR / "final_metrics.json").write_text(json.dumps(final_metrics, indent=2))
print(json.dumps({k: v for k, v in final_metrics.items()
                  if k not in ("sensitivity", "slices", "bootstrap_ci")}, indent=2))

In [ ]:
eps_pass = flip.min() if len(flip) else None
worst_slice = slices.loc[slices["brier"].idxmax()]

card = f"""# Model Card — Telco Customer Churn

**Generated:** {final_metrics['timestamp']}
**Seed:** {RANDOM_SEED}

## Intended Use

Ranks active customers by churn risk so a retention team can decide who receives
a time-limited discount offer. Scored at the start of each monthly billing cycle.

**Not intended for** pricing decisions, credit decisions, or any use where the
score affects a customer's terms rather than whether they are contacted.

## Model

| | |
| :--- | :--- |
| Estimator | {selection['chosen_model']} |
| Calibration | {selection['chosen_calibration']} |
| Threshold | {selection['chosen_threshold_strategy']} — per-customer, derived from cost structure |
| Training rows | {manifest['train_rows']:,} |
| Features | {manifest['n_features']} |

## Performance (held-out test set, evaluated once)

| Metric | Value | 95% CI |
| :--- | ---: | :--- |
| ROC-AUC | {test_metrics['roc_auc']:.4f} | [{ci.loc['roc_auc','lo_2.5']:.4f}, {ci.loc['roc_auc','hi_97.5']:.4f}] |
| Average precision | {test_metrics['average_precision']:.4f} | [{ci.loc['average_precision','lo_2.5']:.4f}, {ci.loc['average_precision','hi_97.5']:.4f}] |
| Log loss | {test_metrics['log_loss']:.4f} | — |
| Brier | {test_metrics['brier']:.4f} | — |
| Expected cost per customer | USD {test_metrics['cost_per_customer']:.2f} | [{ci.loc['cost','lo_2.5']:.2f}, {ci.loc['cost','hi_97.5']:.2f}] |
| Headroom captured | {captured:.1%} | [{cap_lo:.1%}, {cap_hi:.1%}] |

**Verdict against the rule fixed before modelling began (continue at {PASS_AT:.0%},
abandon below {ABORT_BELOW:.0%}): {VERDICT}.** {note}

Cross-validated estimate was {CV_ESTIMATE:.1%}; the test figure differs by
{drift:+.1%} after {selection['validation_queries']} validation queries.

## Limitations

1. **No time dimension.** The dataset is a single snapshot, so out-of-time
   validation is impossible and drift cannot be simulated. Real deployment
   requires monitoring that this project cannot pre-validate.
2. **Predictive, not causal.** The model ranks who is likely to leave. It does
   not establish that contacting them is profitable. Highest-risk customers are
   often the hardest to persuade, and some churn *because* an offer prompted
   them to reconsider.
3. **Cost parameters are assumptions.** Margin, offer cost, horizon, and
   effectiveness are illustrative industry figures, not internal data. Every
   monetary figure above is conditional on them.
4. **The conclusion depends on an unmeasured parameter.** At the assumed
   effectiveness of {COST_ASSUMPTIONS['offer_effectiveness']:.2f} the verdict is
   {VERDICT}. {"Passing would require effectiveness of at least " + f"{eps_pass:.2f}." if eps_pass else "The verdict does not reach PASS anywhere in the tested range."}
5. **Marginal, not conditional, uncertainty guarantees.** Conformal coverage of
   {covered.mean():.1%} holds on average, not within every subgroup.

## Fairness

Calibration parity was chosen as the criterion, because the probability feeds a
monetary decision. Equal calibration and equal error rates cannot both hold when
base rates differ, so this is a stated trade-off rather than a solved problem.

Largest calibration gap among protected attributes: {worst['slice']} at
{worst['calib_gap']:+.4f} (n={int(worst['n'])}).

**Weakest slice by discrimination: {worst_auc['slice']}, ROC-AUC
{worst_auc['roc_auc']:.4f} against {overall_auc:.4f} overall (n={int(worst_auc['n'])},
base rate {worst_auc['base_rate']:.3f}).** This segment has too few churn events to
learn from, yet it carries the largest value at risk per customer, so the model
ranks worst precisely where each error is most expensive. Long-contract
high-bill customers should not be scored by this model without human review.

## Recommendation

Run retention offers as a randomised experiment on a high-risk slice before
scaling. That measures effectiveness directly, converts the largest assumption
in this project into data, and produces the outcome labels needed to train an
uplift model — which is the correct tool for this decision.

## Reproduction

Notebooks 01 through 05 in order, seed {RANDOM_SEED}. Each writes a JSON artifact
consumed by the next. `final_pipeline.joblib` is the scoring artifact; rebuild
from it rather than from parameters recorded in JSON.
"""

(OUT_DIR / "model_card.md").write_text(card)
print(card)

---

## Summary

| Item | Result |
| :--- | :--- |
| Test set | Opened once, after pre-registration |
| Primary metric | Expected cost per customer, derived threshold, no refitting on test |
| Uncertainty | Stratified bootstrap, 2,000 resamples, on every reported figure |
| Regression to the mean | Test versus cross-validated estimate, reported with the query count |
| Error analysis | Ranked by cost incurred rather than by count |
| Slices | Seven attributes, calibration gap and base rate alongside every metric |
| Fairness | Calibration parity chosen and stated; the conflict acknowledged rather than solved |
| Interpretation | Coefficients, permutation importance on test, SHAP |
| Per-prediction uncertainty | Split conformal, coverage checked overall and per slice |
| Sensitivity | Verdict mapped across effectiveness and margin |

**The project is finished at this point.** Any further modelling would make every
number above an in-sample estimate, and would have to be declared as such.

**What remains is the Streamlit app**, which loads `final_pipeline.joblib` and
imports thresholds and costs from `telco_churn.business` — the same code path
evaluated here, so the app cannot silently disagree with the notebook.